# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields, with each referenced by its `@id`.
We will explore what record sets are present to understand how data is organized in the Croissant schema.

In [ ]:
# List available record sets, their IDs, field IDs and columns
from mlcroissant._dataset.metadata import RecordSet

record_sets = metadata.record_sets
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs.name}\n  @id: {rs.id}")
    print("  Fields and their @id's:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("  Columns and their @id's:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print()

## 3. Data Extraction
We will load one or more record sets into pandas DataFrames for analysis.

You must reference record set, field, and column by their `@id`. In this example, we'll demonstrate on the first record set, but you may repeat for others.

In [ ]:
# Extract data from all available record sets for analysis
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"---\nRecord set '{record_set_id}' loaded. Shape: {df.shape}")
    if not df.empty:
        print("Columns:", df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps, such as filtering by a numeric field, normalization, and grouping data. All entity references are made via their Croissant `@id` fields.

We'll demonstrate these on one record set (the first, for example), and you may adapt by changing `selected_record_set_id`, `numeric_field_id`, or `group_field_id` as appropriate.

In [ ]:
# Choose a record set and fields to explore (adjust IDs as needed)
if record_sets:
    selected_record_set = record_sets[0]
    selected_record_set_id = selected_record_set.id
    df = dataframes[selected_record_set_id]

    # Identify numeric fields (by checking pandas inferred types or field metadata)
    # For demo, use the first float/integer column
    numeric_field_id = None
    for field in selected_record_set.fields:
        if field.data_type in ('Number', 'Float', 'Integer') and field.id in df.columns:
            numeric_field_id = field.id
            break
    if numeric_field_id is None:
        # fallback: use the first column with numeric dtype in df
        num_cols = df.select_dtypes(include=[float, int]).columns
        if len(num_cols) > 0:
            numeric_field_id = num_cols[0]

    if numeric_field_id and numeric_field_id in df.columns:
        # Example filter threshold (may need to be adjusted depending on data)
        threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in record set '@id: {selected_record_set_id}' with field '{numeric_field_id}' > {threshold}")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: find first string/categorical column to group by
        group_field_id = None
        for field in selected_record_set.fields:
            if field.data_type == 'Text' and field.id in df.columns:
                group_field_id = field.id
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped means of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No numeric field could be identified for this record set.")
else:
    print("No record sets to analyze.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

For demonstration, we plot the histogram of the analyzed numeric field and, if grouping is possible, a bar chart of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of field '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()
    
if 'grouped_df' in locals() and group_field_id in grouped_df.columns:
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}'")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We demonstrated how to load and explore a Croissant-structured dataset (`mlcroissant`) using only `@id` references.
- You can adapt the notebook to focus on other record sets, fields, and analytic approaches.

**Next steps:** Consider deeper domain analysis, advanced visualization, or exporting processed subsets for downstream ML tasks.